# ***IMPORTS***

In [5]:
#pip install openpyxl
#pip install beautifulsoup4

import openpyxl

from bs4 import BeautifulSoup
import requests
import os
from datetime import datetime
import time
import winsound
#from colorama import init, Fore, Back, Style
#import pandas as pd


import warnings
warnings.filterwarnings('ignore')

## **FUNCIONES Y VARIABLES GLOBALES**

In [6]:
# Conectar y obtener datos desde Zooplus

#Se obtiene el User Agent desde http://httpbin.org/get
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36", "Accept-Encoding":"gzip, deflate", "Accept":"text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8", "DNT":"1","Connection":"close", "Upgrade-Insecure-Requests":"1"}

#Funcion para obtener el precio de un producto
def get_zooplus_price(web_prefix, ean):
    url = web_prefix + ean    
    #Se obtiene el HTML del producto
    page = requests.get(url, headers=headers, verify=False)
    soup1 = BeautifulSoup(page.content, "html.parser")

    #Se extrae el precio del HTML
    try:
        price = soup1.find("span", { "class" : "z-price__amount z-price__amount--standard" }).text.strip() # type: ignore
       
    except:
        ean12 = str(ean).zfill(12)
        url = web_prefix + ean12

        page = requests.get(url, headers=headers, verify=False)
        soup1 = BeautifulSoup(page.content, "html.parser")
        
        try:
            price = soup1.find("span", { "class" : "z-price__amount z-price__amount--standard" }).text.strip() # type: ignore
            
        except:
            ean13 = str(ean).zfill(13)
            url = web_prefix + ean13

            page = requests.get(url, headers=headers, verify=False)
            soup1 = BeautifulSoup(page.content, "html.parser")
            
            try:
                price = soup1.find("span", { "class" : "z-price__amount z-price__amount--standard" }).text.strip() # type: ignore
            
            except:
                url = web_prefix + ean    
    
                page = requests.get(url, headers=headers, verify=False)
                soup1 = BeautifulSoup(page.content, "html.parser")
                
                try:
                    price = soup1.find("span", {"class" : "z-product-price__amount"}).text.strip() # type: ignore
                except:
                    ean12 = str(ean).zfill(12)
                    url = web_prefix + ean12

                    page = requests.get(url, headers=headers, verify=False)
                    soup1 = BeautifulSoup(page.content, "html.parser")
                    try:
                        price = soup1.find("span", {"class" : "z-product-price__amount"}).text.strip() # type: ignore
                    except:
                        ean13 = str(ean).zfill(13)
                        url = web_prefix + ean13

                        page = requests.get(url, headers=headers, verify=False)
                        soup1 = BeautifulSoup(page.content, "html.parser")
                    try:
                        price = soup1.find("span", {"class" : "z-product-price__amount"}).text.strip() # type: ignore
                        
                    except AttributeError:
                        price = "No se ha podido obtener el precio para este producto."
    
    return price

#funciones para hora y fecha

def actual_date():
    now = datetime.now()
    date_now = now.date()
   
    return date_now

def actual_hour():
    now = datetime.now()
    hour_now = now.time()

    return hour_now

def pitido():
    frecuency = 1000
    duration = 100
    winsound.Beep(frecuency, duration)

## **CONFIGURACIÓN DE ARCHIVO DE EXCEL**

In [7]:
#Incluir la localizacion del fichero excel con los datos de Zooplus:
directory = os.getcwd()
path = directory + "\\Precios.xlsx"
#print(directory)

# Se abre el workbook y la hoja seleccionados (donde se guardo el documento por ultima vez)
wb_obj = openpyxl.load_workbook(path)
sheet_obj = wb_obj.active

# Obtener el numero de filas y columnas usadas
row_limit = sheet_obj.max_row # type: ignore
column_limit = sheet_obj.max_column # type: ignore
  
print("Total de filas:", row_limit)
print("Total de columnas:", column_limit)

#Introducir el numero de columna en la que se encuentra el EAN (A=1, B=2, C=3...)
ean_col = 3

#Introducir el numero de la fila en la que se encuentra el primer valor de EAN
first_ean_row = 2

#Introducir el numero de columna en la que se encuentra el precio
precio_col = 4

Total de filas: 22
Total de columnas: 4


## **EJECUTAR TAREA DE RELLENADO DE PRECIOS**

In [8]:
# Cerrar el fichero Excel antes de ejecutar esta celda

#Introducir el formato de la web que se va a buscar
from alive_progress import alive_bar
from alive_progress import animations

web_prefix = "https://www.zooplus.es/search/results?q="

date = actual_date()
hour = actual_hour()
time1 = time.time()

print(("INICIADO EL {} A LAS {} HORAS").format(date, hour))

for i in range(first_ean_row, row_limit + 1): 
    cell_obj = sheet_obj.cell(row = i, column = ean_col) # type: ignore
    ean = str(cell_obj.value)
    
    if ean is None:
        ean = ""
    
    
    #url = web_prefix#+ean
    try:
        price = get_zooplus_price(web_prefix, ean)
        print(("{}: {} tiene un precio de: {}").format(str(i-1).zfill(4), ean, price))
    
        #Se introduce el precio en el Excel
        c1 = sheet_obj.cell(row = i, column = precio_col) # type: ignore
        c1.value = price
    except ConnectionError:
        #Manejo de caida de conexión
        bar = animations.bar_factory('😴', tip="😪", background='zZz', borders=('Durmiendo 👉 ->|','|<- Terminado 🤘'), errors=('<---👀', '💀'))
        print("Hemos tenido un error de conexión, voy a descansar 10 segundos. ZZZZZZzzzzz...")
        total = 10
        i = 0
        
        with alive_bar(total, tittle = "Durmiendo", bar=bar) as bar:
            while i < total:
                time.sleep(1)
                bar()
                i += 1
        
        time.sleep(10)
        print("Buena siesta!!! Vuelvo a la carga!!!!")
        try:
            price = get_zooplus_price(web_prefix, ean)
            print(("{}/{}: {} tiene un precio de: {}").format(str(i-1).zfill(4), (row_limit-1), ean, price))
    
            #Se introduce el precio en el Excel
            c1 = sheet_obj.cell(row = i, column = precio_col) # type: ignore
            c1.value = price
        except:
            break
    
#Se guarda el fichero Excel con los cambios aplicados
wb_obj.save(path)
time2 = time.time()
hour1 = actual_hour()
dif_time = time2 - time1
print(("FINALIZADO EL {} A LAS {} HORAS").format(date, hour1))
print(("REALIZADO EN: {} horas").format((dif_time/60)/60))
print("ARCHIVO GUARDADO / PROGRAMA FINALIZADO")


INICIADO EL 2025-06-18 A LAS 09:16:20.857716 HORAS
0001: 4008429085307 tiene un precio de: 10,49 €
0002: 3065890158306 tiene un precio de: 88,49 €
0003: 4018761211029 tiene un precio de: 19,99 €
0004: 4062911040199 tiene un precio de: 27,99 €
0005: 4062911028852 tiene un precio de: 7,49 €
0006: 4062911018266 tiene un precio de: 12,99 €
0007: 4062911019645 tiene un precio de: 6,99 €
0008: 4054651007099 tiene un precio de: 12,49 €
0009: 4062911019652 tiene un precio de: 6,99 €
0010: 035585356686 tiene un precio de: 5,99 €
0011: 4008239594709 tiene un precio de: 2,19 €
0012: 5900951320408 tiene un precio de: 24,99 €
0013: 4008429112676 tiene un precio de: 14,69 €
0014: 4017721837415 tiene un precio de: 14,79 €
0015: 4008429161926 tiene un precio de: 37,99 €
0016: 4017721869355 tiene un precio de: 2,69 €
0017: 660048001683 tiene un precio de: 4,79 €
0018: 8410650586786 tiene un precio de: 38,09 €
0019: 052742211701 tiene un precio de: 25,99 €
0020: 4054651015377 tiene un precio de: 4,79 €
